# Session 3. From an API to annotated data

**French party manifestos, coded by humans, by an LLM through an API, and by a model on your own machine.**

LLMs for Social Science Research, University of Zurich, 4 November 2026. Course repository: https://github.com/subtilhugo/uzh-llm-socsci-2026

[Open in Colab](https://colab.research.google.com/github/subtilhugo/uzh-llm-socsci-2026/blob/main/sessions/03/s3_manifesto_annotation.ipynb)

| Part | What you do | Time |
| --- | --- | --- |
| 1 | Make your first API request, and read what comes back | 10 min |
| 2 | Collect the French manifestos of one election, and check the collection is complete | 15 min |
| 3 | Define the coding task, draw a sample, **code 15 sentences yourself** | 15 min |
| 4 | Annotate the sample with Claude, through the API | 15 min |
| 5 | Annotate the same sample with a local model (Ollama) | 20 min |
| 6 | Compare: you, the Manifesto Project coders, the API model, the local model | 20 min |
| 7 | Save the reproducibility bundle | 5 min |

**The data.** The Manifesto Project (WZB Berlin) splits each manifesto into quasi-sentences (one statement each) and trained human coders assign each one to one of 56 policy categories. Those codes are our benchmark. They are not the truth: they are one careful coder's reading, and human coders disagree too.

**Keys you need.** A Manifesto Project API key (free, from your account page on manifesto-project.wzb.eu) and an Anthropic API key (the workshop key Hugo sent, capped). Never paste a key into a cell: in Colab, use the key icon in the left sidebar ("Secrets"), with the names `MANIFESTO_API_KEY` and `ANTHROPIC_API_KEY`, and switch on "Notebook access". On your laptop, set them as environment variables.

**No Manifesto key yet?** Part 2 has a fallback: it loads the file Hugo prepared from the course repository.

## 0. Setup

In [ ]:
# Run once. In Colab it takes about a minute.
%pip install -q requests pandas scikit-learn krippendorff anthropic ollama openai pydantic

In [ ]:
import os, json, time, random, hashlib, datetime, pathlib, subprocess
from typing import Literal

import requests
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", 120)


def get_secret(name):
    # Colab Secrets first, then environment variables, then ask (the value is not echoed).
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    from getpass import getpass
    return getpass(f"{name} (not shown, not saved): ")

In [ ]:
# ---- Parameters: everything a referee would ask about is here ----
COUNTRY = "France"
ELECTION_YEAR = 2022          # change after Part 2 shows which elections have annotated texts
N_SAMPLE = 200                # quasi-sentences sent to the models
SEED = 2026                   # sample draw, and local model seed
USE_CONTEXT = True            # show the model the previous quasi-sentence, as human coders read in context

CLAUDE_MODEL = "claude-haiku-5-5"   # a model ID is a pinned snapshot: report it as is
LOCAL_MODEL = "gemma3:4b"           # any model you pulled with Ollama; see the slides for sizes

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M")
ROOT = pathlib.Path("s3_work")                      # in Colab, this lives in the VM and disappears
RAW = ROOT / "data" / "raw" / "manifesto"            # raw API answers, never edited
PROC = ROOT / "data" / "processed"
RUN = ROOT / "outputs" / "annotation" / RUN_ID
for p in (RAW, PROC, RUN):
    p.mkdir(parents=True, exist_ok=True)
print("Run:", RUN)

## Part 1. What an API is

An API (application programming interface) is a door that a data provider built on purpose. You send a **request** to a URL; the server sends back a **response**, usually in JSON.

```
https://manifesto-project.wzb.eu/api/v1/list_core_versions?api_key=XXXX
\______________ base URL ______________/\____ endpoint ____/\_ parameters _/
```

Four things to read in every response: the **status code** (200 is fine; 401 bad key; 404 wrong endpoint; 429 too many requests, wait), the **content**, what is **missing**, and the **version** of the data you got.

Compared to scraping a website: the provider documents the format, keeps it stable, versions the data, and tells you the rules (key, limits, licence). Same logic tomorrow for a parliament's API, a survey archive, a newspaper archive, or an LLM.

In [ ]:
MANIFESTO_KEY = get_secret("MANIFESTO_API_KEY")
BASE = "https://manifesto-project.wzb.eu/api/v1/"


def api_get(endpoint, retries=4, **params):
    # One GET request with a timeout, retries on 429 and 5xx, and an error that never prints the key.
    params = {"api_key": MANIFESTO_KEY, **params}
    for attempt in range(retries):
        r = requests.get(BASE + endpoint, params=params, timeout=60)
        if r.status_code == 429 or r.status_code >= 500:
            wait = 2 ** attempt
            print(f"  {r.status_code}, waiting {wait}s")
            time.sleep(wait)
            continue
        if r.status_code != 200:
            raise RuntimeError(f"{endpoint}: HTTP {r.status_code}: {r.text[:300]}")
        return r.json()
    raise RuntimeError(f"{endpoint}: still failing after {retries} attempts")


# The first request. Read the answer before going on.
versions = api_get("list_core_versions")
print(json.dumps(versions, indent=1)[:800])

In [ ]:
# Which versions exist? Always record the version you used: the data change between releases.
def last_id(obj, list_key):
    items = obj.get(list_key, obj) if isinstance(obj, dict) else obj
    ids = [it["id"] if isinstance(it, dict) else it for it in items]
    return ids[-1], ids

CORE_VERSION, _ = last_id(versions, "datasets")
CORPUS_VERSION, _ = last_id(api_get("list_metadata_versions"), "versions")
print("Core dataset:", CORE_VERSION, "| corpus:", CORPUS_VERSION)

## Part 2. Collect the manifestos of one French election

Three requests: the **core dataset** (one row per party and election, with the coded shares), the **metadata** of each document (language, annotated or not), and the **texts with their codes**.

Then the question from last week: *the API returned the documents and the code says it is complete. How would you know?* Four checks, written into the code.

**Option A** (you have a Manifesto key): run the cells below. **Option B** (no key yet): run only the next cell, then go to Part 3.

In [ ]:
# OPTION B ONLY. The table Hugo built tonight with the cells below (Manifesto Project data, CC BY-NC-SA 4.0).
url = "https://raw.githubusercontent.com/subtilhugo/uzh-llm-socsci-2026/main/sessions/03/data/manifesto_fr_quasisentences.csv"
qs = pd.read_csv(url, dtype={"cmp_code": str})
CORE_VERSION, CORPUS_VERSION = qs["core_version"].iloc[0], qs["corpus_version"].iloc[0]
print(len(qs), "quasi-sentences;", CORE_VERSION, CORPUS_VERSION)

In [ ]:
core_json = api_get("get_core", key=CORE_VERSION)
core = pd.DataFrame(core_json[1:], columns=core_json[0])     # first row is the header
fr = core[core["countryname"] == COUNTRY].copy()
fr["year"] = fr["date"].astype(str).str[:4].astype(int)
print(len(core), "party-election rows in the core dataset;", len(fr), "for", COUNTRY)
fr.groupby("year").agg(parties=("party", "nunique"), names=("partyabbrev", lambda s: ", ".join(sorted(map(str, s))))).tail(8)

In [ ]:
# The documents of the chosen election, and whether they are annotated.
el = fr[fr["year"] == ELECTION_YEAR].copy()
keys = [f"{p}_{d}" for p, d in zip(el["party"], el["date"])]
meta = api_get("metadata", **{"keys[]": keys, "version": CORPUS_VERSION})
meta_df = pd.DataFrame(meta["items"])
show = [c for c in ["manifesto_id", "party_id", "language", "annotations", "title"] if c in meta_df.columns]
print("Requested:", len(keys), "| metadata returned:", len(meta_df), "| missing:", meta.get("missing_items"))
meta_df[show]

In [ ]:
# Texts and codes. The raw answer is archived first, before any processing.
annotated = meta_df.loc[meta_df["annotations"].astype(str).str.lower().isin(["true", "1"]), "manifesto_id"].tolist()
res = api_get("texts_and_annotations", **{"keys[]": annotated, "version": CORPUS_VERSION})
raw_file = RAW / f"texts_{COUNTRY}_{ELECTION_YEAR}_{CORPUS_VERSION}.json"
raw_file.write_text(json.dumps(res, ensure_ascii=False))
print("Archived:", raw_file, f"({raw_file.stat().st_size/1e6:.1f} MB)")

rows = []
for doc in res["items"]:
    for i, it in enumerate(doc["items"]):
        rows.append({"key": doc["key"], "pos": i, "text": it.get("text", ""),
                     "cmp_code": str(it.get("cmp_code", "NA"))})
qs = pd.DataFrame(rows)
qs["party"] = qs["key"].str.split("_").str[0].astype(int)
qs = qs.merge(el[["party", "partyabbrev", "partyname"]].astype({"party": int}), on="party", how="left")
print(len(qs), "quasi-sentences in", qs["key"].nunique(), "documents")
qs.head(8)

In [ ]:
# The four completeness checks.
returned = {d["key"] for d in res["items"]}
missing = set(res.get("missing_items", []))

# 1. Every document we asked for came back, or is listed as missing. Nothing silently dropped.
assert returned | missing == set(annotated), "a requested document is neither returned nor listed as missing"
print("1. Requested", len(annotated), "| returned", len(returned), "| listed as missing", len(missing))

# 2. No empty text, no empty code.
assert qs["text"].str.strip().ne("").all(), "empty quasi-sentence"
assert qs["cmp_code"].ne("").all(), "empty code"
print("2. No empty text or code")

# 3. The number of coded quasi-sentences matches the 'total' published in the core dataset.
coded = qs[~qs["cmp_code"].isin(["H", "NA", "nan"])]
n_by_doc = coded.groupby("party").size().rename("from_texts")
check = el.set_index("party")[["partyabbrev", "total"]].join(n_by_doc)
check["diff"] = check["from_texts"] - check["total"]
print("3. Quasi-sentences per document, our count against the published total:")
display(check)

# 4. Recompute one published share from the texts: per504 (welfare state expansion).
rec = (100 * coded["cmp_code"].str.split(".").str[0].eq("504").groupby(coded["party"]).mean()).rename("per504_texts")
cmp504 = el.set_index("party")[["partyabbrev", "per504"]].join(rec)
print("4. per504 recomputed from the texts against the core dataset (should match closely):")
display(cmp504.round(2))

**Read the four checks before going on.** If a difference is not zero, find out why before you use the data (headings, uncoded sentences, a document revised between versions). This is the answer to last week's question: completeness is checked against **a number the provider published independently of your request**, not against the agent's report.

In [ ]:
# INSTRUCTOR: export the table used as the fallback (Option B), with the versions in the file itself.
qs.assign(core_version=CORE_VERSION, corpus_version=CORPUS_VERSION).to_csv(
    PROC / "manifesto_fr_quasisentences.csv", index=False)
print("Wrote", PROC / "manifesto_fr_quasisentences.csv")

## Part 3. The coding task, and your own coding first

The 56 categories are hard, for humans too. We start with their **seven domains** (the first digit of the code), plus one class for statements with no policy content. Exercise 4 goes back to the 56.

| Domain | Codes | Examples of categories |
| --- | --- | --- |
| 1 External relations | 101–110 | military, peace, internationalism, the EU |
| 2 Freedom and democracy | 201–204 | human rights, democracy, the constitution |
| 3 Political system | 301–305 | decentralisation, efficiency of government, corruption, political authority |
| 4 Economy | 401–416 | free market, regulation, planning, protectionism, growth, technology |
| 5 Welfare and quality of life | 501–507 | environment, culture, equality, welfare state, education |
| 6 Fabric of society | 601–608 | national way of life, traditional morality, law and order, multiculturalism |
| 7 Social groups | 701–706 | workers, farmers, middle class, minorities, women, the elderly, the young |
| 0 No category | 000 | no policy content |

In [ ]:
DOMAINS = {
    "external_relations": 1, "freedom_democracy": 2, "political_system": 3, "economy": 4,
    "welfare_quality_of_life": 5, "fabric_of_society": 6, "social_groups": 7, "no_category": 0,
}
NUM2DOM = {v: k for k, v in DOMAINS.items()}

qs = qs.sort_values(["key", "pos"]).reset_index(drop=True)
qs["prev_text"] = qs.groupby("key")["text"].shift(1).fillna("")
work = qs[~qs["cmp_code"].isin(["H", "NA", "nan"])].copy()          # headings are not coded
work["human_domain"] = work["cmp_code"].str[0].astype(int).map(NUM2DOM)

print(pd.crosstab(work["human_domain"], work["partyabbrev"], margins=True))

In [ ]:
# The sample: the same number of sentences per party, drawn with a fixed seed.
per_party = N_SAMPLE // work["party"].nunique()
parts = [d.sample(min(per_party, len(d)), random_state=SEED) for _, d in work.groupby("party")]
sample = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample["item_id"] = sample["key"] + ":" + sample["pos"].astype(str)
sample[["item_id", "partyabbrev", "cmp_code", "human_domain", "text"]].to_csv(RUN / "sample.csv", index=False)
print(len(sample), "quasi-sentences;", sample["partyabbrev"].value_counts().to_dict())

### Before any model: code 15 sentences yourself (10 minutes)

Read each sentence (and the one before it), choose a domain from the table above, and write it in `MY_CODES`, in order. Do not look at the human codes. You are the third coder: at the end we will see whom you agree with.

In [ ]:
for i, r in sample.head(15).iterrows():
    print(f"[{i}] ({r.partyabbrev})  previous: {r.prev_text[:90]}")
    print(f"     TARGET: {r.text}\n")

In [ ]:
# One domain name per sentence, in order: external_relations, freedom_democracy, political_system,
# economy, welfare_quality_of_life, fabric_of_society, social_groups, no_category
MY_CODES = [
    # "economy",
]

## Part 4. Annotate through an API

An LLM API works like the Manifesto API: a request with a key, a JSON response. What we send is the **codebook** (system prompt), **one quasi-sentence** (user message), and a **schema** the answer must follow, so the output is machine-readable without parsing free text.

Three things to know about Claude 5.5 models through the API:
- The model ID (`claude-haiku-5-5`) is a pinned snapshot: same weights for the life of the ID. Report it.
- `temperature` and `seed` are not accepted. Runs can differ: we will measure it (Part 6) rather than assume it away.
- The codebook is identical in every request: it is **cached**, and cache reads cost a tenth of the input price.

In [ ]:
CODEBOOK = '''You are an expert coder for the Manifesto Project (MARPOR). You code quasi-sentences of party
manifestos into policy domains. A quasi-sentence is one statement; it can be a fragment of a sentence.
The manifestos are in French; reason about the meaning, not the words.

Assign exactly one domain to the TARGET quasi-sentence:
- external_relations: relations with other countries, foreign special relationships, anti-imperialism,
  the military, peace, internationalism, the European Union (positive or negative).
- freedom_democracy: freedom and human rights, democracy, constitutionalism (positive or negative).
- political_system: decentralisation or centralisation, governmental and administrative efficiency,
  political corruption, political authority (strong government, competence of the party).
- economy: free market, incentives, market regulation, economic planning, corporatism, protectionism,
  economic goals, demand management, economic growth, technology and infrastructure, controlled economy,
  nationalisation, economic orthodoxy and budget discipline, Marxist analysis, anti-growth economy.
- welfare_quality_of_life: environmental protection, culture, equality and social justice, welfare state
  expansion or limitation, education expansion or limitation.
- fabric_of_society: national way of life (positive or negative), traditional morality (positive or
  negative), law and order, civic mindedness, multiculturalism (positive or negative).
- social_groups: labour groups and workers, agriculture and farmers, middle class and professional
  groups, underprivileged minority groups, non-economic demographic groups (women, the elderly, the young).
- no_category: no policy content (greetings, pure rhetoric, statements that cannot be coded).

Rules: code the TARGET only; the PREVIOUS quasi-sentence is context. If a statement fits several
domains, choose the one its main policy content belongs to. Give a one-sentence reason in English,
then the domain.'''


class Annotation(BaseModel):
    reason: str = Field(description="One short sentence, in English")
    domain: Literal["external_relations", "freedom_democracy", "political_system", "economy",
                    "welfare_quality_of_life", "fabric_of_society", "social_groups", "no_category"]


def user_message(row):
    prev = f"PREVIOUS: {row['prev_text']}\n" if USE_CONTEXT and row["prev_text"] else ""
    return f"{prev}TARGET: {row['text']}"


print(user_message(sample.iloc[0]))

In [ ]:
import anthropic

claude = anthropic.Anthropic(api_key=get_secret("ANTHROPIC_API_KEY"), max_retries=5)


def annotate_claude(row, model=CLAUDE_MODEL):
    r = claude.messages.parse(
        model=model,
        max_tokens=300,
        system=[{"type": "text", "text": CODEBOOK, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user_message(row)}],
        output_format=Annotation,
        thinking={"type": "disabled"},   # a short, well-defined task: no thinking, low cost
    )
    out = r.parsed_output
    return {"domain": out.domain if out else None, "reason": out.reason if out else None,
            "stop_reason": r.stop_reason, "model_returned": r.model,
            "in": r.usage.input_tokens, "out": r.usage.output_tokens,
            "cache_read": r.usage.cache_read_input_tokens or 0,
            "cache_write": r.usage.cache_creation_input_tokens or 0}


# One call. Look at everything that comes back before you run 200 of them.
annotate_claude(sample.iloc[0])

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def run(annotator, df, name, workers=4):
    # Annotate every row once. Answers are appended to a JSONL file as they arrive, so a crash or
    # a rerun never pays twice for the same item.
    path = RUN / f"raw_{name}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text().splitlines():
            rec = json.loads(line)
            done[rec["item_id"]] = rec
    todo = [r for _, r in df.iterrows() if r["item_id"] not in done]

    def one(row):
        t0 = time.time()
        try:
            rec = annotator(row)
        except Exception as e:
            rec = {"domain": None, "error": repr(e)[:300]}
        rec.update(item_id=row["item_id"], annotator=name, seconds=round(time.time() - t0, 2),
                   utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
        return rec

    with ThreadPoolExecutor(max_workers=workers) as pool, path.open("a") as f:
        for i, rec in enumerate(pool.map(one, todo), 1):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            done[rec["item_id"]] = rec
            if i % 25 == 0:
                print(f"  {name}: {i}/{len(todo)}")
    out = pd.DataFrame([done[i] for i in df["item_id"]])
    print(f"{name}: {out['domain'].notna().sum()}/{len(out)} annotated, "
          f"{out.get('error', pd.Series(dtype=str)).notna().sum()} errors")
    return out


res_claude = run(annotate_claude, sample, "claude")

In [ ]:
# What did it cost? Prices per million tokens for Claude Haiku 5.5 (prompts under 100k tokens), November 2026.
PRICE = {"in": 0.10, "out": 0.50, "cache_read": 0.01, "cache_write": 0.125}
tok = res_claude[["in", "out", "cache_read", "cache_write"]].sum()
cost = sum(tok[k] * PRICE[k] for k in PRICE) / 1e6
print(tok.to_dict())
print(f"Cost of {len(res_claude)} annotations: ${cost:.4f}; "
      f"for the {len(work):,} quasi-sentences of this election: about ${cost / len(res_claude) * len(work):.2f}")

## Part 5. The same task with a local model

A local model runs on your machine: nothing leaves it (confidential data), it costs nothing per call, and you can pin the exact weights for a replication. It is smaller, so usually less accurate, and slower without a good GPU.

**Ollama** downloads, serves and runs open-weight models. Two ways today:
- **On your laptop**: install from https://ollama.com/download, then in a terminal `ollama pull gemma3:4b`. Run this notebook locally (VS Code, Cursor or Jupyter), not in Colab: Colab cannot see your laptop.
- **In Colab**: Runtime, Change runtime type, **T4 GPU**. Then run the next cell, which installs Ollama inside the Colab machine.

In [ ]:
# COLAB ONLY: install and start Ollama inside the Colab machine (GPU runtime). Skip on your laptop.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", LOCAL_MODEL], check=True)

In [ ]:
import ollama

# Record exactly which weights we use: name, digest, quantization. A tag like 'gemma3:4b' can be updated;
# the digest cannot.
info = {m.model: m for m in ollama.list().models}
assert LOCAL_MODEL in info, f"{LOCAL_MODEL} not pulled. In a terminal: ollama pull {LOCAL_MODEL}"
LOCAL_DIGEST = info[LOCAL_MODEL].digest
det = ollama.show(LOCAL_MODEL).details
print(LOCAL_MODEL, "| digest", LOCAL_DIGEST[:12], "| parameters", det.parameter_size, "| quantization", det.quantization_level)


def annotate_local(row, model=LOCAL_MODEL):
    r = ollama.chat(
        model=model,
        messages=[{"role": "system", "content": CODEBOOK}, {"role": "user", "content": user_message(row)}],
        format=Annotation.model_json_schema(),          # the same schema as for Claude
        options={"temperature": 0, "seed": SEED, "num_ctx": 4096},
    )
    out = Annotation.model_validate_json(r.message.content)
    return {"domain": out.domain, "reason": out.reason,
            "in": r.prompt_eval_count, "out": r.eval_count}


annotate_local(sample.iloc[0])

In [ ]:
# One worker: a laptop runs one request at a time anyway.
res_local = run(annotate_local, sample, "local_" + LOCAL_MODEL.replace(":", "_"), workers=1)
print("Mean seconds per sentence:", res_local["seconds"].mean().round(2))

**Any other server that speaks the OpenAI format** works with the same code: LM Studio on your laptop, vLLM or Ollama on the ScienceCluster, or a commercial API. Only `base_url`, the key and the model name change.

In [ ]:
from openai import OpenAI

SCHEMA = Annotation.model_json_schema() | {"additionalProperties": False}


def make_openai_annotator(base_url, model, api_key="not-needed"):
    client = OpenAI(base_url=base_url, api_key=api_key)

    def annotate(row):
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": CODEBOOK}, {"role": "user", "content": user_message(row)}],
            response_format={"type": "json_schema", "json_schema": {"name": "annotation", "schema": SCHEMA, "strict": True}},
            temperature=0, seed=SEED,
        )
        out = Annotation.model_validate_json(r.choices[0].message.content)
        return {"domain": out.domain, "reason": out.reason,
                "in": r.usage.prompt_tokens, "out": r.usage.completion_tokens}
    return annotate


# Examples (uncomment one):
# annotate_oa = make_openai_annotator("http://localhost:11434/v1", LOCAL_MODEL)          # Ollama, OpenAI format
# annotate_oa = make_openai_annotator("http://localhost:1234/v1", "<model in LM Studio>")  # LM Studio
# annotate_oa = make_openai_annotator("http://localhost:8000/v1", "<model>")               # vLLM through an SSH tunnel
# res_oa = run(annotate_oa, sample, "openai_compatible")

## Part 6. Validate

Agreement with the Manifesto Project coders is the first number, not the last. Read it with three others: how stable the model is when asked twice, how much **you** agree with the coders, and whether the **aggregate** a paper would report (a party's share of welfare statements) survives the disagreements.

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

LABELS = list(DOMAINS)
results = {"claude": res_claude}
if "res_local" in globals():
    results["local"] = res_local
if "res_oa" in globals():
    results["openai_compatible"] = res_oa

eval_df = sample[["item_id", "partyabbrev", "text", "human_domain"]].copy()
for name, res in results.items():
    eval_df[name] = res.set_index("item_id").loc[eval_df["item_id"], "domain"].values


def scores(y_true, y_pred):
    ok = pd.notna(y_pred)
    yt, yp = np.asarray(y_true)[ok], np.asarray(y_pred)[ok]
    return {"n": int(ok.sum()), "accuracy": accuracy_score(yt, yp),
            "macro_F1": f1_score(yt, yp, labels=LABELS, average="macro", zero_division=0),
            "kappa": cohen_kappa_score(yt, yp, labels=LABELS)}


table = pd.DataFrame({name: scores(eval_df["human_domain"], eval_df[name]) for name in results}).T
if len(results) >= 2:
    a, b = list(results)[:2]
    table.loc[f"{a} vs {b}"] = scores(eval_df[a], eval_df[b])
table.round(3)

In [ ]:
# You, against the Manifesto coders and against the models, on your 15 sentences.
if MY_CODES:
    mine = eval_df.head(len(MY_CODES)).assign(me=MY_CODES)
    print({"you vs human coder": round(accuracy_score(mine["human_domain"], mine["me"]), 2),
           **{f"you vs {n}": round(accuracy_score(mine[n].fillna("none"), mine["me"]), 2) for n in results}})
else:
    print("Fill MY_CODES in Part 3 first.")

In [ ]:
# Where do the disagreements fall? Rows: Manifesto coders. Columns: the API model.
pd.crosstab(eval_df["human_domain"], eval_df["claude"], margins=True)

In [ ]:
# Stability: the same 30 sentences, a second time. Without temperature control, how many answers change?
second = run(annotate_claude, sample.head(30), "claude_rerun")
same = (second["domain"].values == res_claude.head(30)["domain"].values).mean()
print(f"Identical answers on the rerun: {same:.0%}")

In [ ]:
# The aggregate a paper would report: each party's share of quasi-sentences per domain, coders against models.
# With 200 sentences these shares are noisy; Exercise 2 runs a whole manifesto.
agg = {}
for col in ["human_domain"] + list(results):
    agg[col] = pd.crosstab(eval_df["partyabbrev"], eval_df[col], normalize="index").reindex(columns=LABELS, fill_value=0)
for name in results:
    r = np.corrcoef(agg["human_domain"].values.ravel(), agg[name].values.ravel())[0, 1]
    print(f"Correlation of party x domain shares, human vs {name}: {r:.2f}")
(100 * agg["human_domain"]).round(0).astype(int)

In [ ]:
# Ten disagreements, with the model's reason. For each one: who is right, the coder or the model?
dis = eval_df[eval_df["human_domain"] != eval_df["claude"]].head(10)
dis = dis.merge(res_claude[["item_id", "reason"]], on="item_id")
for _, r in dis.iterrows():
    print(f"({r.partyabbrev}) {r.text}\n   coder: {r.human_domain} | model: {r.claude} | reason: {r.reason}\n")

## Part 7. The reproducibility bundle

What a referee needs to rerun this, and what goes in your supplementary material: the data versions, the sample, the exact prompt and schema, the models and their parameters, every raw answer, and the scores. Download the folder (Colab: Files panel, right click) or let it live in your project folder.

In [ ]:
(RUN / "prompt_codebook.txt").write_text(CODEBOOK)
(RUN / "schema.json").write_text(json.dumps(Annotation.model_json_schema(), indent=2))
params = {
    "run_id": RUN_ID, "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "data": {"source": "Manifesto Project (WZB) API", "core_version": CORE_VERSION,
             "corpus_version": CORPUS_VERSION, "country": COUNTRY, "election_year": ELECTION_YEAR},
    "sample": {"n": len(sample), "seed": SEED, "stratified_by": "party", "use_context": USE_CONTEXT},
    "api_model": {"provider": "Anthropic", "model": CLAUDE_MODEL, "thinking": "disabled",
                  "structured_output": "Annotation schema", "temperature": "not settable on this model"},
    "local_model": {"runtime": "Ollama", "model": LOCAL_MODEL, "digest": globals().get("LOCAL_DIGEST"),
                    "quantization": getattr(globals().get("det"), "quantization_level", None),
                    "temperature": 0, "seed": SEED},
    "prompt_sha256": hashlib.sha256(CODEBOOK.encode()).hexdigest(),
}
(RUN / "params.json").write_text(json.dumps(params, indent=2))
table.round(4).to_csv(RUN / "scores.csv")
eval_df.to_csv(RUN / "annotations.csv", index=False)
print(sorted(p.name for p in RUN.iterdir()))

## Exercises (until the end of the block, then in your own project)

1. **Change one thing in the prompt**: definitions, two examples per domain (few-shot), `USE_CONTEXT = False`, a codebook in French. Rerun Parts 4 and 6 under a new `RUN_ID`. Did kappa move? Paste *name, change, kappa before → after* in the shared document.
2. **A whole manifesto.** Estimate the cost first (Part 4 prints it), then annotate every quasi-sentence of one party and compare its domain shares to the published ones in the core dataset.
3. **A second local model.** `ollama pull` another one (the slides list sizes), rerun Part 5: speed against kappa.
4. **The 56 categories.** Ask for `cmp_code` instead of the domain. How far does agreement fall, and is it further than human coders fall?

**Your own project**: prompt 10 (collect from an API, with completeness checks) and prompt 11 (annotate with an LLM, validate against your own coding) in `template/prompts/`.

*Data: Lehmann, P. et al. Manifesto Project Dataset and Corpus, WZB. Cite the version printed in `params.json`.*